In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nih-chest-xrays/data")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/nih-chest-xrays/data


In [2]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import densenet121, DenseNet121_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score

In [3]:
import os

DATA_DIR = "/kaggle/input/datasets/nih-chest-xrays/data"

CSV_PATH = os.path.join(
    DATA_DIR,
    "Data_Entry_2017.csv"
)

IMAGE_DIRS = [
    os.path.join(DATA_DIR, f"images_{i:03d}", "images")
    for i in range(1, 13)
]

print("CSV:", CSV_PATH)

for directory in IMAGE_DIRS:
    print(directory, "->", os.path.exists(directory))

CSV: /kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv
/kaggle/input/datasets/nih-chest-xrays/data/images_001/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_002/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_003/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_004/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_005/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_006/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_007/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_008/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_009/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_010/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_011/images -> True
/kaggle/input/datasets/nih-chest-xrays/data/images_012/images -> True


In [4]:
image_paths = {}

for image_dir in IMAGE_DIRS:
    for image_name in os.listdir(image_dir):
        image_paths[image_name] = os.path.join(
            image_dir,
            image_name
        )

print("Total images found:", len(image_paths))

Total images found: 112120


In [5]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (112120, 12)


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [6]:
missing_images = [
    image_name
    for image_name in df["Image Index"]
    if image_name not in image_paths
]

print("Missing images:", len(missing_images))

if len(missing_images) > 0:
    print(missing_images[:10])

Missing images: 0


In [7]:
from torch.utils.data import Dataset
from PIL import Image
import torch
import numpy as np


class ChestXrayDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_paths,
        transform=None
    ):

        self.dataframe = dataframe.reset_index(drop=True)
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_name = row["Image Index"]

        image_path = self.image_paths[image_name]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(
            row[diseases].values.astype(np.float32)
        )

        return image, labels

In [8]:
classes = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Hernia",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax",
    "No Finding"
]

print("Number of classes:", len(classes))

Number of classes: 15


In [9]:
for class_name in classes:
    df[class_name] = df["Finding Labels"].apply(
        lambda x: 1 if class_name in x.split("|") else 0
    )

In [10]:
print(df[classes].sum().sort_values())

Hernia                  227
Pneumonia              1431
Fibrosis               1686
Edema                  2303
Emphysema              2516
Cardiomegaly           2776
Pleural_Thickening     3385
Consolidation          4667
Pneumothorax           5302
Mass                   5782
Nodule                 6331
Atelectasis           11559
Effusion              13317
Infiltration          19894
No Finding            60361
dtype: int64


In [11]:
from sklearn.model_selection import train_test_split

patients = df["Patient ID"].unique()

train_patients, temp_patients = train_test_split(
    patients,
    test_size=0.20,
    random_state=42
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=42
)

train_df = df[df["Patient ID"].isin(train_patients)].copy()
val_df = df[df["Patient ID"].isin(val_patients)].copy()
test_df = df[df["Patient ID"].isin(test_patients)].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 89826
Validation: 10930
Test: 11364


In [12]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [13]:
from torch.utils.data import Dataset
from PIL import Image
import torch
import numpy as np

class ChestXrayDataset(Dataset):

    def __init__(self, dataframe, image_paths, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image_name = row["Image Index"]
        image_path = self.image_paths[image_name]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(
            row[classes].values.astype(np.float32)
        )

        return image, labels

In [14]:
train_dataset = ChestXrayDataset(
    train_df,
    image_paths,
    train_transform
)

val_dataset = ChestXrayDataset(
    val_df,
    image_paths,
    val_transform
)

test_dataset = ChestXrayDataset(
    test_df,
    image_paths,
    val_transform
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 89826
Validation: 10930
Test: 11364


In [15]:
image, label = train_dataset[0]

print("Image shape:", image.shape)
print("Label shape:", label.shape)
print("Labels:", label)

Image shape: torch.Size([3, 224, 224])
Label shape: torch.Size([15])
Labels: tensor([0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])


In [17]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("DataLoaders recreated successfully.")
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

DataLoaders recreated successfully.
Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32, 15])


In [18]:
images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32, 15])


# Load pretrained DenseNet121

In [19]:
import torch
import torch.nn as nn
from torchvision.models import densenet121, DenseNet121_Weights

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

weights = DenseNet121_Weights.DEFAULT

model = densenet121(weights=weights)

Device: cuda
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 222MB/s]


## Replace the classifier

In [20]:
num_features = model.classifier.in_features

model.classifier = nn.Linear(
    num_features,
    15
)

model = model.to(device)

print(model.classifier)

Linear(in_features=1024, out_features=15, bias=True)


# Calculate class weights

In [21]:
positive_counts = train_df[classes].sum().values

negative_counts = len(train_df) - positive_counts

pos_weights = negative_counts / positive_counts

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float32
).to(device)

for class_name, weight in zip(classes, pos_weights):
    print(f"{class_name:20s}: {weight.item():.2f}")

Atelectasis         : 8.61
Cardiomegaly        : 40.62
Consolidation       : 23.17
Edema               : 47.58
Effusion            : 7.35
Emphysema           : 44.25
Fibrosis            : 66.95
Hernia              : 500.82
Infiltration        : 4.69
Mass                : 18.09
Nodule              : 16.78
Pleural_Thickening  : 31.96
Pneumonia           : 78.00
Pneumothorax        : 19.98
No Finding          : 0.86


## Loss function

In [ ]:
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights
)

## Optimizer

In [22]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

## Learning-rate scheduler

In [23]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

# Training function

In [24]:
from tqdm.auto import tqdm

def train_one_epoch(model, loader, criterion, optimizer):

    model.train()

    running_loss = 0.0

    progress_bar = tqdm(loader, desc="Training")

    for images, labels in progress_bar:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda"):

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return running_loss / len(loader)

# Validation function

In [25]:
def validate(model, loader, criterion):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for images, labels in tqdm(
            loader,
            desc="Validation"
        ):

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with autocast("cuda"):

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )

            probabilities = torch.sigmoid(outputs)

            running_loss += loss.item()

            all_labels.append(
                labels.cpu().numpy()
            )

            all_probs.append(
                probabilities.cpu().numpy()
            )

    all_labels = np.concatenate(all_labels)
    all_probs = np.concatenate(all_probs)

    auc_scores = []

    for i, class_name in enumerate(classes):

        auc = roc_auc_score(
            all_labels[:, i],
            all_probs[:, i]
        )

        auc_scores.append(auc)

    mean_auc = np.mean(auc_scores)

    return (
        running_loss / len(loader),
        mean_auc,
        all_labels,
        all_probs
    )

In [26]:
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

print("Model device:", next(model.parameters()).device)

Train batches: 2808
Validation batches: 342
Test batches: 356
Model device: cuda:0


In [27]:
from torch.amp import autocast, GradScaler

scaler = GradScaler("cuda")

In [29]:
import torch
import torch.nn as nn

# Loss function
criterion = nn.CrossEntropyLoss()

print("Criterion created successfully:", criterion)

Criterion created successfully: CrossEntropyLoss()


In [30]:
train_loss = train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer
)

val_loss, val_auc, _, _ = validate(
    model,
    val_loader,
    criterion
)

print()
print(f"Train Loss: {train_loss:.4f}")
print(f"Val Loss:   {val_loss:.4f}")
print(f"Val AUC:    {val_auc:.4f}")

Training:   0%|          | 0/2808 [00:00<?, ?it/s]

Validation:   0%|          | 0/342 [00:00<?, ?it/s]


Train Loss: 2.1592
Val Loss:   2.0470
Val AUC:    0.7771


In [31]:
EPOCHS = 8

best_auc = 0.0
patience = 3
epochs_without_improvement = 0

history = {
    "train_loss": [],
    "val_loss": [],
    "val_auc": []
}

BEST_MODEL_PATH = "/kaggle/working/best_densenet121.pth"

print("Training configuration ready.")
print("Epochs:", EPOCHS)
print("Best model path:", BEST_MODEL_PATH)

Training configuration ready.
Epochs: 8
Best model path: /kaggle/working/best_densenet121.pth


# Training function with AMP

In [33]:
from tqdm.auto import tqdm
from torch.amp import autocast, GradScaler

scaler = GradScaler("cuda")


def train_one_epoch(model, loader, criterion, optimizer):

    model.train()

    running_loss = 0.0

    progress_bar = tqdm(
        loader,
        desc="Training"
    )

    for images, labels in progress_bar:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with autocast("cuda"):

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return running_loss / len(loader)

# Validation function

In [34]:
from sklearn.metrics import roc_auc_score


def validate(model, loader, criterion):

    model.eval()

    running_loss = 0.0

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for images, labels in tqdm(
            loader,
            desc="Validation"
        ):

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with autocast("cuda"):

                outputs = model(images)

                loss = criterion(
                    outputs,
                    labels
                )

            probabilities = torch.sigmoid(
                outputs
            )

            running_loss += loss.item()

            all_labels.append(
                labels.cpu().numpy()
            )

            all_probs.append(
                probabilities.cpu().numpy()
            )

    all_labels = np.concatenate(
        all_labels
    )

    all_probs = np.concatenate(
        all_probs
    )

    auc_scores = []

    for i in range(len(classes)):

        auc = roc_auc_score(
            all_labels[:, i],
            all_probs[:, i]
        )

        auc_scores.append(auc)

    mean_auc = np.mean(auc_scores)

    return (
        running_loss / len(loader),
        mean_auc,
        all_labels,
        all_probs
    )

# Train + save the best model

In [35]:
for epoch in range(EPOCHS):

    print("\n" + "=" * 60)
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print("=" * 60)

    # -------------------------
    # Training
    # -------------------------

    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    # -------------------------
    # Validation
    # -------------------------

    val_loss, val_auc, _, _ = validate(
        model,
        val_loader,
        criterion
    )

    # -------------------------
    # Scheduler
    # -------------------------

    scheduler.step(val_auc)

    # -------------------------
    # Save history
    # -------------------------

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        val_loss
    )

    history["val_auc"].append(
        val_auc
    )

    # -------------------------
    # Results
    # -------------------------

    print(f"\nTrain Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val ROC-AUC: {val_auc:.4f}")

    # -------------------------
    # Save best model
    # -------------------------

    if val_auc > best_auc:

        best_auc = val_auc
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_auc": val_auc,
                "classes": classes
            },
            BEST_MODEL_PATH
        )

        print("🔥 BEST MODEL SAVED!")

    else:

        epochs_without_improvement += 1

        print(
            f"No improvement: "
            f"{epochs_without_improvement}/{patience}"
        )

        if epochs_without_improvement >= patience:

            print("\n⛔ Early stopping triggered.")
            break


Epoch 1/8


Training:   0%|          | 0/2808 [00:00<?, ?it/s]

Validation:   0%|          | 0/342 [00:00<?, ?it/s]


Train Loss : 2.0402
Val Loss   : 2.0316
Val ROC-AUC: 0.7919
🔥 BEST MODEL SAVED!

Epoch 2/8


Training:   0%|          | 0/2808 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/342 [00:00<?, ?it/s]


Train Loss : 1.7876
Val Loss   : 2.0149
Val ROC-AUC: 0.8062
No improvement: 2/3

Epoch 8/8


Training:   0%|          | 0/2808 [00:00<?, ?it/s]

Validation:   0%|          | 0/342 [00:00<?, ?it/s]


Train Loss : 1.7366
Val Loss   : 2.0467
Val ROC-AUC: 0.8072
No improvement: 3/3

⛔ Early stopping triggered.
